# Demo-Prep - Run Once (full end-to-end, multi-site)

One notebook that refreshes the **entire demo** from the seeded stream all the way to Power BI,
across **every site** (the reference site plus any synthetic sites created by `Multi-Site-Fanout`):

1. **PI-Seed-Synthetic** (backfill) - brings raw `PiEvents` (KQL) current to *now* for **all tags/all sites**.
2. **Verify** - prints live tag coverage **per plant** so you can confirm every site is streaming key tags.
3. **PI-Gold-Delta** - appends the new events into **`gold.fact_pi`** (the Direct Lake table Power BI reads).
4. **Watchlist-Seed-Synthetic** - regenerates today's `ml.watchlist` + `ml.daily_narrative` + dashboard spotlight.
5. **Refresh** - syncs the `lh_poc` SQL endpoint metadata and refreshes the semantic model(s).
6. **PI-Seed-Synthetic** (stream) - streams live values for `STREAM_HOURS` so dashboards keep advancing.

All workspace item ids (child notebooks, semantic models, SQL endpoint, eventhouse) are **resolved
by name at runtime**, so this works in any deployment with no hand-editing.

**Run all.** The demo is usable once step 5 finishes (~5-10 min); the notebook then stays running for
the live stream. Set `STREAM_HOURS = 0` for backfill-only (no live stream).


In [ ]:
# PARAMETERS  (Fabric: this cell is tagged 'parameters')
RUN_PI_BACKFILL = True     # 1) PI-Seed-Synthetic: backfill PiEvents (KQL) up to now (all sites)
VERIFY_SITES    = True     # 2) print per-plant live tag coverage so you can confirm every site streams
RUN_GOLD        = True     # 3) PI-Gold-Delta: append new PiEvents -> gold.fact_pi (the gold table PBI reads)
RUN_WATCHLIST   = True     # 4) Watchlist-Seed-Synthetic: ml.watchlist + ml.daily_narrative + dashboard spotlight
RUN_REFRESH     = True     # 5) refresh lh_poc SQL endpoint metadata + semantic model(s)
STREAM_HOURS    = 2.0      # 6) live-stream PiEvents for this many hours (0 = no live stream); runs LAST
STEP_TIMEOUT_SEC = 3600    # per-notebook timeout for backfill / gold / watchlist steps
VERIFY_WINDOW_MIN = 30     # look-back window (minutes) for the per-plant live coverage check


In [ ]:
# Resolve every workspace id we need BY NAME at runtime (deployment-agnostic).
import notebookutils, time, json, requests, re

FAB = "https://api.fabric.microsoft.com/v1"

def _cred():
    try:
        import notebookutils as _n; return _n.credentials
    except Exception:
        from notebookutils import mssparkutils as _m; return _m.credentials

def _fabric_token():
    c = _cred()
    for aud in ("pbi", "https://analysis.windows.net/powerbi/api", "https://api.fabric.microsoft.com"):
        try:
            t = c.getToken(aud)
            if t: return t
        except Exception:
            pass
    raise RuntimeError("could not acquire a Fabric API token")

def _hdr():
    return {"Authorization": "Bearer " + _fabric_token(), "Content-Type": "application/json"}

def current_ws():
    # notebook runtime context first, then env fallback
    try:
        ctx = notebookutils.runtime.context
        for k in ("currentWorkspaceId", "workspaceId"):
            v = ctx.get(k) if hasattr(ctx, "get") else None
            if v: return v
    except Exception:
        pass
    try:
        from notebookutils import mssparkutils
        v = mssparkutils.env.getWorkspaceId()
        if v: return v
    except Exception:
        pass
    raise RuntimeError("could not determine current workspace id")

def list_items(ws, itemtype=None):
    url = f"{FAB}/workspaces/{ws}/items" + (f"?type={itemtype}" if itemtype else "")
    out = []
    while url:
        r = requests.get(url, headers=_hdr())
        if r.status_code != 200:
            raise RuntimeError(f"list items failed {r.status_code}: {r.text[:200]}")
        j = r.json(); out += j.get("value", [])
        url = j.get("continuationUri")
    return out

WS_ID = current_ws()
ITEMS = list_items(WS_ID)

def find_id(itemtype, exact=None, pattern=None):
    for i in ITEMS:
        if i.get("type") != itemtype:
            continue
        name = i.get("displayName", "")
        if exact is not None and name == exact:
            return i["id"]
        if pattern is not None and re.search(pattern, name, re.I):
            return i["id"]
    return None

NB_IDS = {n: find_id("Notebook", exact=n) for n in ("PI-Seed-Synthetic", "PI-Gold-Delta", "Watchlist-Seed-Synthetic")}

# Semantic models: refresh whichever exist (import model updates the report; Direct Lake auto-reframes).
DS_IDS = {n: find_id("SemanticModel", exact=n) for n in ("semantic-main-import", "semantic-main")}
DS_IDS = {n: i for n, i in DS_IDS.items() if i}

def sql_endpoint_id(ws):
    r = requests.get(f"{FAB}/workspaces/{ws}/lakehouses", headers=_hdr())
    for lh in (r.json().get("value", []) if r.status_code == 200 else []):
        if lh.get("displayName") == "lh_poc":
            return ((lh.get("properties", {}) or {}).get("sqlEndpointProperties", {}) or {}).get("id")
    return None

def eventhouse_query_uri(ws):
    r = requests.get(f"{FAB}/workspaces/{ws}/eventhouses", headers=_hdr())
    for e in (r.json().get("value", []) if r.status_code == 200 else []):
        u = (e.get("properties", {}) or {}).get("queryServiceUri")
        if u: return u
    return None

SQL_EP    = sql_endpoint_id(WS_ID)
KUSTO_URI = eventhouse_query_uri(WS_ID)

print("Workspace:", WS_ID)
print("Child notebooks:", {k: (v[:8] + '...' if v else None) for k, v in NB_IDS.items()})
print("Semantic models:", list(DS_IDS.keys()))
print("SQL endpoint:", (SQL_EP[:8] + '...') if SQL_EP else None, "| Eventhouse URI:", KUSTO_URI)

missing = [k for k, v in NB_IDS.items() if not v]
if missing:
    raise RuntimeError(f"could not resolve child notebook(s) by name: {missing} - are they deployed to this workspace?")


# --- Demo status writer: publishes a small JSON to the lakehouse so the deploy
#     wizard can show live stage progress (best-effort; never fails the run). ---
LH_STATUS_PATH = f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com/lh_poc.Lakehouse/Files/demo-status.json"
def set_status(step, stage, label, ready=False, error=None):
    try:
        payload = {"step": step, "total": 6, "stage": stage, "label": label,
                   "ready": bool(ready), "error": error,
                   "ts": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
        notebookutils.fs.put(LH_STATUS_PATH, json.dumps(payload), True)
    except Exception as e:
        print("  (status write skipped:", str(e)[:120], ")", flush=True)


In [ ]:
def run_job(step, name, params=None, poll_secs=15, max_polls=800):
    """Submit a child notebook as an isolated job (fresh Spark session) and wait. Raises on failure."""
    nid = NB_IDS[name]
    body = {}
    if params:
        body = {"executionData": {"parameters": {k: {"value": str(v), "type": "float"} for k, v in params.items()}}}
    print(f"\n[{step}] submit {name} params={params or {}} ...", flush=True)
    ts = time.time()
    r = requests.post(f"{FAB}/workspaces/{WS_ID}/items/{nid}/jobs/instances?jobType=RunNotebook",
                      headers=_hdr(), data=json.dumps(body))
    if r.status_code not in (200, 201, 202):
        raise RuntimeError(f"{name}: submit failed {r.status_code} {r.text[:300]}")
    loc = r.headers.get("Location")
    for _ in range(max_polls):
        time.sleep(poll_secs)
        s = requests.get(loc, headers=_hdr())
        st = (s.json() or {}).get("status")
        if st in ("Completed", "Deduped"):
            print(f"[{step}] {name} OK in {round(time.time()-ts)}s", flush=True); return
        if st in ("Failed", "Cancelled"):
            fr = (s.json() or {}).get("failureReason")
            raise RuntimeError(f"{name} {st}: {json.dumps(fr)[:400]}")
    raise RuntimeError(f"{name}: timed out")

# --- KQL helper for the multi-site verification (best-effort) ---
def kusto_query(csl):
    if not KUSTO_URI:
        raise RuntimeError("no eventhouse query uri")
    c = _cred(); tok = None
    for aud in (KUSTO_URI, "kusto", "pbi"):
        try:
            tok = c.getToken(aud)
            if tok: break
        except Exception:
            pass
    if not tok:
        raise RuntimeError("no kusto token")
    import ssl, urllib.request
    ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    body = json.dumps({"db": "pi-realtime-db", "csl": csl}).encode()
    req = urllib.request.Request(f"{KUSTO_URI}/v1/rest/query", data=body, method="POST",
        headers={"Authorization": "Bearer " + tok, "Content-Type": "application/json", "Accept": "application/json"})
    with urllib.request.urlopen(req, context=ctx) as resp:
        j = json.loads(resp.read().decode())
    tbl = next((t for t in j.get("Tables", []) if t.get("Rows")), j.get("Tables", [{}])[0] if j.get("Tables") else {})
    cols = [c["ColumnName"] for c in tbl.get("Columns", [])]
    return [dict(zip(cols, row)) for row in tbl.get("Rows", [])]

def verify_sites(window_min):
    csl = (f"PiEvents | where Ts > ago({int(window_min)}m) and isnotempty(Plant) "
           f"| summarize tags=dcount(Tag), points=count(), lastTs=max(Ts) by Plant | order by Plant asc")
    rows = kusto_query(csl)
    print(f"\n--- Live tag coverage per plant (last {int(window_min)} min) ---", flush=True)
    if not rows:
        print("  (no recent PiEvents rows - run the backfill first, or widen VERIFY_WINDOW_MIN)", flush=True)
        return
    total_plants = len(rows); total_tags = sum(int(r.get("tags") or 0) for r in rows)
    for r in rows:
        print(f"  {str(r.get('Plant')):<14} tags={str(r.get('tags') or 0):<5} points={str(r.get('points') or 0):<8} lastTs={r.get('lastTs')}", flush=True)
    print(f"  => {total_plants} plant(s) live, {total_tags} distinct tags streaming across all sites", flush=True)


In [ ]:
t0 = time.time()
print("=== Demo-Prep (full end-to-end, run once, multi-site) ===", flush=True)
set_status(0, "starting", "Starting demo prep\u2026")

try:
    # 1) Backfill raw PiEvents (KQL) up to now for ALL tags/sites. STREAM_HOURS=0 => backfill only.
    if RUN_PI_BACKFILL:
        set_status(1, "backfill", "Seeding sensor history (backfill) \u2014 the longest step\u2026")
        run_job("1/6", "PI-Seed-Synthetic", {"STREAM_HOURS": 0.0})
    else:
        print("[1/6] PI-Seed backfill skipped")

    # 2) Confirm every site is streaming its key tags (per-plant live coverage).
    if VERIFY_SITES:
        set_status(2, "verify", "Verifying live coverage across all sites\u2026")
        try:
            verify_sites(VERIFY_WINDOW_MIN)
        except Exception as e:
            print("[2/6] site verification skipped:", str(e)[:160], flush=True)
    else:
        print("[2/6] site verification skipped")

    # 3) Roll the new raw events into gold.fact_pi (the Delta table the Power BI Direct Lake model reads).
    if RUN_GOLD:
        set_status(3, "gold", "Rolling new events into the gold table\u2026")
        run_job("3/6", "PI-Gold-Delta")
    else:
        print("[3/6] PI-Gold-Delta skipped")

    # 4) Regenerate today's watchlist + daily narrative and refresh the dashboard spotlight.
    if RUN_WATCHLIST:
        set_status(4, "watchlist", "Refreshing watchlist & daily narrative\u2026")
        run_job("4/6", "Watchlist-Seed-Synthetic")
    else:
        print("[4/6] Watchlist-Seed skipped")

    # 5) Refresh SQL endpoint metadata + semantic model(s) so Power BI shows the new data.
    if RUN_REFRESH:
        set_status(5, "refresh", "Refreshing SQL endpoint & semantic models\u2026")
        print("\n[5/6] Refresh SQL endpoint metadata + semantic model(s) ...", flush=True)
        if SQL_EP:
            try:
                r = requests.post(f"{FAB}/workspaces/{WS_ID}/sqlEndpoints/{SQL_EP}/refreshMetadata?preview=true",
                                  headers=_hdr(), data="{}", timeout=180)
                print("    SQL endpoint refreshMetadata:", r.status_code, flush=True)
            except Exception as e:
                print("    SQL endpoint refreshMetadata skipped:", str(e)[:150], flush=True)
        else:
            print("    SQL endpoint not resolved - skipping metadata refresh", flush=True)
        for nm, ds in DS_IDS.items():
            try:
                r = requests.post(f"https://api.powerbi.com/v1.0/myorg/groups/{WS_ID}/datasets/{ds}/refreshes",
                                  headers=_hdr(), data=json.dumps({"type": "full"}), timeout=120)
                print(f"    semantic model '{nm}' refresh trigger:", r.status_code, flush=True)
            except Exception as e:
                print(f"    semantic model '{nm}' refresh skipped:", str(e)[:150], flush=True)
    else:
        print("[5/6] Refresh skipped")

    # Data is seeded and current -> the web app is ready to launch NOW (before the long stream).
    set_status(6, "ready", "Data seeded \u2014 your real-time feed is starting. You can launch the web app now.", ready=True)
    print(f"\n=== Demo is READY (elapsed {round(time.time()-t0)}s). gold.fact_pi, watchlist, narrative and dashboards are current for all sites. ===", flush=True)

    # 6) Optional: keep dashboards live by streaming fresh values for STREAM_HOURS. Runs LAST (waits for the duration).
    if STREAM_HOURS and float(STREAM_HOURS) > 0:
        set_status(6, "streaming", f"Live feed running \u2014 streaming fresh data (~{float(STREAM_HOURS):g}h). You can launch the web app.", ready=True)
        run_job("6/6", "PI-Seed-Synthetic", {"STREAM_HOURS": float(STREAM_HOURS)},
                poll_secs=30, max_polls=int(float(STREAM_HOURS) * 3600 / 30) + 60)
        set_status(6, "done", "Live stream finished. Re-run Launch Demo any time to stream again.", ready=True)
        print(f"=== Live stream finished. Total elapsed {round(time.time()-t0)}s. ===", flush=True)
    else:
        set_status(6, "done", "Demo data is ready (static as of now).", ready=True)
        print("[6/6] Live stream disabled (STREAM_HOURS=0). Demo data is static as of now.", flush=True)
except Exception as _e:
    set_status(-1, "error", "Demo prep failed: " + str(_e)[:160], ready=False, error=str(_e)[:300])
    raise
